In [1]:
import time
from pathlib import Path

import pandas as pd
import requests

In [2]:
queries = [
    "SAP",
    "SAP AI",
    "SAP ERP",
    '"S/4HANA"',
    "enterprise software"
]

hits = []

for q in queries:

    url = f"https://hn.algolia.com/api/v1/search?query={q}&hitsPerPage=50"

    response = requests.get(url)
    time.sleep(0.2)
    data = response.json()

    hits.extend(data.get("hits", []))

len(hits)

222

In [3]:
# search returns both stories and comments, so the fields differ a bit
# between hit types - comments don't have a title, stories don't have comment_text
hn_docs = []

for hit in hits:

    title = hit.get("title") or hit.get("story_title") or ""
    text = hit.get("comment_text") or hit.get("story_text") or ""

    hn_docs.append({
        "id": hit["objectID"],
        "title": title,
        "description": "",
        "content": text,
        "published": hit.get("created_at", ""),
        "source": "Hacker News",
        "url": hit.get("url") or f"https://news.ycombinator.com/item?id={hit['objectID']}"
    })

hn_df = pd.DataFrame(hn_docs)
len(hn_df)

222

In [4]:
print("Before:", len(hn_df))

hn_df = hn_df.drop_duplicates(subset="id")
hn_df = hn_df[(hn_df["title"] != "") | (hn_df["content"] != "")]
hn_df = hn_df.drop(columns="id").reset_index(drop=True)

print("After:", len(hn_df))

Before: 222
After: 220


In [5]:
data_dir = Path("../notebook/data")
if not data_dir.exists():
    data_dir = Path("notebook/data")
data_dir.mkdir(parents=True, exist_ok=True)

hn_df.to_json(data_dir / "hackernews_posts.json", orient="records", indent=2)

print("saved", len(hn_df), "hackernews posts to", data_dir / "hackernews_posts.json")

saved 220 hackernews posts to ../notebook/data/hackernews_posts.json
